In [14]:
import os
from statistics import mean
from itertools import product
import numpy as np
from collections import defaultdict
import itertools

In [15]:
def informacion(dir,dic):
    fp = open(dir,'r')
    flag = False
    info = []
    for line in fp:
        aux0 = line.strip().split('(')
        aux1 = aux0[1].split(')')
        line = [aux0[0]]+aux1
        aux =line[1].strip(' ').split(' ')
        config = ''
        for param in aux:
            cont = 0
            while len(param)!=3:
                param = '0'+param
                cont+=1
                '''
                if cont>3:
                    print(dir,aux)
                    exit
                '''
            config+=param
        if config not in dic:
            dic[config] = set()
        dic[config].add(float(line[2].strip(' ')))
        if flag:
            info.append([init,config])
            init=config
        else:
            init=config
            flag = True
    fp.close()
    #print(info)
    return info,dic

In [16]:
def dividir_intervalo(x, y, z):
    return min(x // y, z)

In [17]:
from statistics import mean

def precision(dic, next):
    aprox = [30, 1, 1, 50, 50, 20]  # Definir la granularidad de cada parámetro

    # Normalizar dic (sacar el promedio de cada lista de valores)
    for key in dic:
        dic[key] = mean(dic[key])

    dic2 = {}  # Diccionario para almacenar las configuraciones generadas dinámicamente
    dic3 = {}  # Diccionario para mapear claves originales a discretizadas

    # Convertir claves de `dic` a una representación más precisa
    for key in dic:
        aux = [int(key[i:i+3]) for i in range(0, len(key), 3)]  # Extraer los valores de cada parámetro
        config = " ".join(str(param // aprox[idx]) for idx, param in enumerate(aux))  # Discretizar

        dic3[key] = config  # Guardar la relación clave original -> clave reducida
        if config not in dic2:
            dic2[config] = []  # Crear clave solo si es necesaria
        dic2[config].append(dic[key])

    # Generar `next_new` con claves discretizadas
    next_new = []
    for info in next:
        next_new.append([])
        for nodoi, nodof in info:
            auxi = [int(nodoi[i:i+3]) for i in range(0, len(nodoi), 3)]
            auxf = [int(nodof[i:i+3]) for i in range(0, len(nodof), 3)]

            configi = " ".join(str(param // aprox[idx]) for idx, param in enumerate(auxi))
            configf = " ".join(str(param // aprox[idx]) for idx, param in enumerate(auxf))

            aux1 = "".join(x.zfill(3) for x in configi.split())  # Asegurar formato de tres dígitos
            aux2 = "".join(x.zfill(3) for x in configf.split())

            next_new[-1].append([aux1, aux2])

    # Filtrar y formatear `dic2` para `dic2_new`
    dic2_new = { "".join(x.zfill(3) for x in key.split()): min(values) for key, values in dic2.items() if values }

    return next_new, dic2_new


In [18]:
def escritura(dir,dic,info):
    print('vamo a escribir')
    print(dir)
    fp = open(dir,'a')
    fp.write('Run Fitness1 Solution1 Fitness2 Solution2\n')
    txt = '{} {} {} {} {}\n'
    i=1
    for run in info:
        for pair in run:
            fp.write(txt.format(i,str(dic[pair[0]]), pair[0], str(dic[pair[1]]), pair[1]))
        i+=1
    fp.close()



In [19]:
patho_ = 'all_hyb_20_procesado.txt'

# Lista de archivos a procesar
files = [
    'all_hyb_20.out'
]

# Procesar cada archivo
for file in files:
    dic = {}
    next_data = []

    print(f"Procesando archivo: {file}")

    # Procesa el archivo 10 veces
    for s in range(1):

        path = file
        info, dic = informacion(path, dic)
        next_data.append(info)
    #Llama a precision
    next_data, dic = precision(dic, next_data)

    # Escribe los resultados acumulados en el archivo de salida usando `escritura`
    escritura(patho_, dic, next_data)
    print(f"Resultados del archivo {file} escritos en: {patho_}")

Procesando archivo: all_hyb_20.out
vamo a escribir
all_hyb_20_procesado.txt
Resultados del archivo all_hyb_20.out escritos en: all_hyb_20_procesado.txt


In [20]:
import re

#Codigo para generar un log con todas las locaciones junto a las configuraciones que se encuentran dentro

def process_configurations(file_path, divisor_list):
    # Crear un diccionario para almacenar los resultados
    config_dict = {}

    # Leer el archivo línea por línea
    with open(file_path, 'r') as file:
        for line in file:
            # Buscar configuraciones en el formato (X X X X X X)
            match = re.search(r'\((.*?)\)', line)
            if match:
                config = match.group(1)  # Extraer la configuración como string
                config_values = list(map(int, config.split()))  # Convertir los valores a enteros

                # Aplicar la división entera a cada valor usando la lista de divisores
                if len(config_values) == len(divisor_list):
                    divided_values = [val // div if div != 0 else 0 for val, div in zip(config_values, divisor_list)]

                    # Convertir la configuración dividida a tupla para usarla como clave
                    divided_config = tuple(divided_values)

                    # Almacenar en el diccionario
                    if divided_config not in config_dict:
                        config_dict[divided_config] = []

                    # Agregar la línea completa como valor asociado a la configuración dividida
                    config_dict[divided_config].append(line.strip())
                else:
                    print(f"La configuración {config} no coincide con la longitud de la lista de divisores.")

    return config_dict


file_path = 'all_hyb_20.out'
divisor_list = [10, 1, 1, 20, 20, 10]  # Vector de presicion


result = process_configurations(file_path, divisor_list)

output_file_path = 'locaciones.txt'
with open(output_file_path, 'w') as output_file:
    for divided_config, lines in result.items():
        output_file.write(f"Configuración Dividida: {divided_config}\n")
        for line in lines:
            output_file.write(f"  {line}\n")
        output_file.write("\n")


In [21]:
import collections

# Formateo para aumentar el contador de Run
archivo_superior = "all_hyb_20_procesado.txt"
archivo_inferior = "all_hyb_20.out"
archivo_salida = "all_hyb_final_20.txt"


conteo_numeros = collections.OrderedDict()
with open(archivo_inferior, "r") as f:
    lineas_abajo = f.readlines()

for linea in lineas_abajo:
    partes = linea.split()
    if partes:
        num = int(partes[0])
        conteo_numeros[num] = conteo_numeros.get(num, 0) + 1

with open(archivo_superior, "r") as f:
    lineas_arriba = f.readlines()

nueva_data = [lineas_arriba[0]]  # Mantener la primera línea sin cambios
linea_actual = 0  # Empezamos en la línea 2 del archivo superior

for num_original, cantidad in conteo_numeros.items():
    num_corregido = num_original + 1  # Ajustamos la numeración
    for _ in range(cantidad):  # Modificamos las siguientes 'cantidad' líneas
        if linea_actual < len(lineas_arriba):
            partes = lineas_arriba[linea_actual].split()
            if partes:
                partes[0] = str(num_corregido)  # Reemplazar el número en la línea
                nueva_data.append(" ".join(partes) + "\n")
            linea_actual += 1

# 4. Guardar el resultado
if len(nueva_data) > 1:
    del nueva_data[1]
with open(archivo_salida, "w") as f:
    f.writelines(nueva_data)

for archivo in os.listdir():
    if archivo not in [archivo_inferior, archivo_salida]:
        try:
            os.remove(archivo)
            print(f"Archivo {archivo} eliminado.")
        except Exception as e:
            print(f"No se pudo eliminar {archivo}: {e}")

print(f"Archivo modificado guardado en {archivo_salida}")


No se pudo eliminar .config: [Errno 21] Is a directory: '.config'
Archivo all_hyb_20_procesado.txt eliminado.
No se pudo eliminar .ipynb_checkpoints: [Errno 21] Is a directory: '.ipynb_checkpoints'
Archivo locaciones.txt eliminado.
No se pudo eliminar sample_data: [Errno 21] Is a directory: 'sample_data'
Archivo modificado guardado en all_hyb_final_20.txt
